<a href="https://colab.research.google.com/github/TriambigaKrubhakaran/_VITECHAbusiveTextDravidianLangtech2026_/blob/main/Qwen_Qwen2_5_7B_LoRA_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Tamil Abusive Language Detection using LoRA Fine-tuning
Model: Qwen/Qwen2.5-7B
Task: Binary Classification (Abusive vs Non-Abusive)

MODIFIED VERSION:
- Training: 100% of dataset 1 + 20% for validation
- Prediction: 100% of dataset 2
Dataset: https://docs.google.com/spreadsheets/d/1cjL_qfIRvsSy8mbi1vZP61ThjFj2150pCAwBfEsxdcw
Prediction Dataset: https://docs.google.com/spreadsheets/d/1r5p4CX9oeQqauyFmkI-nZg3nCDRy4mQqa-XoPm0liRg

OPTIMIZED FOR QWEN 2.5 - Compatible with all transformers versions
REQUIRES: Google Colab with GPU (T4 or better)
"""

# ============================================================================
# INSTALLATION
# ============================================================================
print("Installing required packages...")
!pip install -q transformers datasets peft accelerate bitsandbytes scikit-learn pandas huggingface_hub
print("✓ Installation complete!\n")

# ============================================================================
# IMPORTS
# ============================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset
import json
from google.colab import drive
from huggingface_hub import login
import warnings
import gc
import time as time_module
warnings.filterwarnings('ignore')

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

# ============================================================================
# HUGGING FACE AUTHENTICATION (OPTIONAL FOR QWEN)
# ============================================================================
print("="*70)
print("HUGGING FACE AUTHENTICATION (OPTIONAL)")
print("="*70)
print("\nQwen2.5 models are publicly available.")
print("Authentication is optional but recommended for faster downloads.")
print("Get your token from: https://huggingface.co/settings/tokens")
print("="*70 + "\n")

hf_token_input = input("Enter your Hugging Face token (or press Enter to skip): ").strip()
if hf_token_input:
    try:
        login(token=hf_token_input)
        print("✓ Successfully authenticated with Hugging Face!")
        hf_token = hf_token_input
    except Exception as e:
        print(f"⚠ Authentication failed: {e}")
        print("  Continuing without authentication...")
        hf_token = None
else:
    print("✓ Continuing without authentication (public access)")
    hf_token = None

# ============================================================================
# MOUNT GOOGLE DRIVE
# ============================================================================
print("\n" + "="*70)
print("MOUNTING GOOGLE DRIVE")
print("="*70 + "\n")
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")

# ============================================================================
# LOAD TRAINING DATASET FROM GOOGLE SHEETS
# ============================================================================
print("\n" + "="*70)
print("LOADING TRAINING DATASET FROM GOOGLE SHEETS")
print("="*70 + "\n")

training_sheet_url = "https://docs.google.com/spreadsheets/d/1cjL_qfIRvsSy8mbi1vZP61ThjFj2150pCAwBfEsxdcw/export?format=csv&gid=337990985"

df_train_full = pd.read_csv(training_sheet_url)

print(f"✓ Training dataset loaded: {len(df_train_full)} samples")
print(f"✓ Columns: {df_train_full.columns.tolist()}\n")
print("First 5 rows:")
print(df_train_full.head())

# ============================================================================
# LOAD PREDICTION DATASET FROM GOOGLE SHEETS
# ============================================================================
print("\n" + "="*70)
print("LOADING PREDICTION DATASET FROM GOOGLE SHEETS")
print("="*70 + "\n")

prediction_sheet_url = "https://docs.google.com/spreadsheets/d/1r5p4CX9oeQqauyFmkI-nZg3nCDRy4mQqa-XoPm0liRg/export?format=csv&gid=1762241428"

df_predict = pd.read_csv(prediction_sheet_url)

print(f"✓ Prediction dataset loaded: {len(df_predict)} samples")
print(f"✓ Columns: {df_predict.columns.tolist()}\n")
print("First 5 rows:")
print(df_predict.head())

# ============================================================================
# DATA PREPROCESSING FOR TRAINING DATASET
# ============================================================================
print("\n" + "="*70)
print("PREPROCESSING TRAINING DATASET")
print("="*70 + "\n")

# Clean column names
df_train_full.columns = df_train_full.columns.str.strip()

# Identify text and label columns
text_col = 'Text'
label_col = 'Class'

if text_col not in df_train_full.columns or label_col not in df_train_full.columns:
    print(f"⚠ Expected columns: 'Text' and 'Class'")
    print(f"⚠ Found columns: {df_train_full.columns.tolist()}")
    text_col = df_train_full.columns[0]
    label_col = df_train_full.columns[1] if len(df_train_full.columns) > 1 else df_train_full.columns[0]

print(f"✓ Text column: '{text_col}'")
print(f"✓ Label column: '{label_col}'")

# Rename columns
df_train_full = df_train_full.rename(columns={text_col: 'text', label_col: 'label'})
df_train_full = df_train_full[['text', 'label']].copy()

# Clean text data
df_train_full['text'] = df_train_full['text'].astype(str).str.strip()
df_train_full = df_train_full[df_train_full['text'] != '']
df_train_full = df_train_full[df_train_full['text'].notna()]

print(f"\nOriginal label distribution:")
print(df_train_full['label'].value_counts())

# Convert labels to binary
df_train_full['label'] = df_train_full['label'].astype(str).str.strip()

label_map = {}
for label in df_train_full['label'].unique():
    label_str = str(label).lower().strip()
    if 'non' in label_str and 'abuse' in label_str:
        label_map[label] = 0
    elif label_str in ['non-abusive', 'non abusive', 'not abusive', 'clean', 'neutral']:
        label_map[label] = 0
    elif 'abuse' in label_str and 'non' not in label_str:
        label_map[label] = 1
    elif label_str in ['abusive', 'offensive', 'toxic', 'hate']:
        label_map[label] = 1
    else:
        label_map[label] = 0

print(f"\nLabel mapping:")
for original, mapped in sorted(label_map.items(), key=lambda x: x[1]):
    print(f"  '{original}' → {mapped} ({'Abusive' if mapped == 1 else 'Non-Abusive'})")

df_train_full['label'] = df_train_full['label'].map(label_map).astype(int)
df_train_full = df_train_full.dropna(subset=['text', 'label'])
df_train_full = df_train_full[df_train_full['label'].isin([0, 1])]
df_train_full = df_train_full.reset_index(drop=True)

print(f"\n✓ Cleaned training dataset: {len(df_train_full)} samples")
print(f"\nClass balance:")
print(f"  Non-Abusive (0): {(df_train_full['label'] == 0).sum()} ({(df_train_full['label'] == 0).sum()/len(df_train_full)*100:.2f}%)")
print(f"  Abusive (1): {(df_train_full['label'] == 1).sum()} ({(df_train_full['label'] == 1).sum()/len(df_train_full)*100:.2f}%)")

# Calculate class imbalance ratio
class_counts = df_train_full['label'].value_counts()
imbalance_ratio = class_counts[0] / class_counts[1] if class_counts[1] > 0 else 1.0
print(f"\n⚠️ Class imbalance ratio: {imbalance_ratio:.2f}:1 (Non-Abusive:Abusive)")
if imbalance_ratio > 2.0:
    print(f"  → Will apply class weighting (weight for Abusive class: {imbalance_ratio:.2f})")

# ============================================================================
# DATA PREPROCESSING FOR PREDICTION DATASET
# ============================================================================
print("\n" + "="*70)
print("PREPROCESSING PREDICTION DATASET")
print("="*70 + "\n")

# Clean column names
df_predict.columns = df_predict.columns.str.strip()

# Identify text column (prediction dataset may not have labels)
if 'Text' in df_predict.columns:
    pred_text_col = 'Text'
elif 'text' in df_predict.columns:
    pred_text_col = 'text'
else:
    pred_text_col = df_predict.columns[0]

print(f"✓ Prediction text column: '{pred_text_col}'")

# Store original dataframe for later
df_predict_original = df_predict.copy()

# Prepare prediction data
df_predict = df_predict[[pred_text_col]].copy()
df_predict = df_predict.rename(columns={pred_text_col: 'text'})
df_predict['text'] = df_predict['text'].astype(str).str.strip()
df_predict = df_predict[df_predict['text'] != '']
df_predict = df_predict[df_predict['text'].notna()]
df_predict = df_predict.reset_index(drop=True)

print(f"✓ Cleaned prediction dataset: {len(df_predict)} samples")

# ============================================================================
# DATASET SPLIT: 100% TRAINING + 20% VALIDATION (FROM SAME DATA)
# ============================================================================
print("\n" + "="*70)
print("DATASET SPLIT (100% FOR TRAINING + 20% FOR VALIDATION)")
print("="*70 + "\n")

# Use 100% for training
train_df = df_train_full.copy()

# Create 20% validation set from the same data
class_counts = df_train_full['label'].value_counts()
min_class_count = class_counts.min()

if min_class_count < 2:
    val_df = df_train_full.sample(frac=0.2, random_state=42)
else:
    val_df = df_train_full.groupby('label', group_keys=False).apply(
        lambda x: x.sample(frac=0.2, random_state=42)
    )

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Training: {len(train_df)} samples (100% of dataset)")
print(f"  - Non-Abusive: {(train_df['label'] == 0).sum()}")
print(f"  - Abusive: {(train_df['label'] == 1).sum()}")
print(f"\nValidation: {len(val_df)} samples (20% sampled from training data)")
print(f"  - Non-Abusive: {(val_df['label'] == 0).sum()}")
print(f"  - Abusive: {(val_df['label'] == 1).sum()}")
print(f"\n⚠️ Note: Validation set overlaps with training set for maximum training data utilization")

# ============================================================================
# MODEL AND TOKENIZER LOADING WITH 4-BIT QUANTIZATION
# ============================================================================
print("\n" + "="*70)
print("LOADING MODEL AND TOKENIZER")
print("="*70 + "\n")

model_name = "Qwen/Qwen2.5-7B"
print(f"Model: {model_name}")
print("Configuration: 4-bit quantization (NF4) for GPU efficiency\n")

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True,
    use_fast=True
)

# Qwen models have proper pad tokens, but let's ensure it's set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("✓ Tokenizer loaded")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  Pad token: {tokenizer.pad_token}")

# Load model with 4-bit quantization
print("\nLoading model (this may take 2-3 minutes)...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Configure model for binary classification
model.config.pad_token_id = tokenizer.pad_token_id
model.config.id2label = {0: "Non-Abusive", 1: "Abusive"}
model.config.label2id = {"Non-Abusive": 0, "Abusive": 1}

print(f"✓ Model loaded with 4-bit quantization")
print(f"✓ Total parameters: {model.num_parameters():,}")

# Check GPU memory
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    allocated = torch.cuda.memory_allocated(0) / 1e9
    print(f"✓ GPU Memory: {allocated:.1f}GB / {mem_gb:.1f}GB")

# ============================================================================
# LORA CONFIGURATION - OPTIMIZED FOR QWEN 2.5
# ============================================================================
print("\n" + "="*70)
print("CONFIGURING LORA (OPTIMIZED FOR QWEN 2.5)")
print("="*70 + "\n")

# Determine optimal LoRA rank based on dataset size
dataset_size = len(train_df)
if dataset_size < 5000:
    lora_r = 8
    lora_alpha = 16
    print(f"Dataset size: {dataset_size} → Using r=8 (conservative)")
elif dataset_size < 20000:
    lora_r = 16
    lora_alpha = 32
    print(f"Dataset size: {dataset_size} → Using r=16 (recommended)")
else:
    lora_r = 32
    lora_alpha = 64
    print(f"Dataset size: {dataset_size} → Using r=32 (high capacity)")

# QWEN-SPECIFIC: Target modules for Qwen2.5 architecture
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    inference_mode=False
)

model = get_peft_model(model, lora_config)
print("\n✓ LoRA applied to Qwen2.5 architecture")
model.print_trainable_parameters()

# ============================================================================
# DATA TOKENIZATION - OPTIMIZED (DYNAMIC PADDING)
# ============================================================================
print("\n" + "="*70)
print("TOKENIZING DATASETS (DYNAMIC PADDING)")
print("="*70 + "\n")

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=512  # Qwen2.5 supports longer contexts efficiently
    )

# Tokenize training and validation sets
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['text', 'label']])

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("✓ Training and validation datasets tokenized")
print(f"  Training: {len(train_dataset)} samples")
print(f"  Validation: {len(val_dataset)} samples")

# Tokenize prediction dataset (no labels)
predict_dataset = Dataset.from_pandas(df_predict[['text']])
predict_dataset = predict_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
predict_dataset.set_format('torch', columns=['input_ids', 'attention_mask'])

print(f"  Prediction: {len(predict_dataset)} samples")
print(f"  Max length: 512 tokens (optimized for Qwen2.5)")
print(f"  ✅ Using DataCollatorWithPadding for efficient batching")

# ============================================================================
# WEIGHTED TRAINER FOR CLASS IMBALANCE
# ============================================================================

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# Calculate class weights if imbalanced
if imbalance_ratio > 2.0:
    class_weights = torch.tensor([1.0, imbalance_ratio])
    print(f"\n✓ Using class weights: [1.0, {imbalance_ratio:.2f}]")
else:
    class_weights = None
    print("\n✓ Classes are balanced, no weighting needed")

# ============================================================================
# TRAINING CONFIGURATION - OPTIMIZED FOR QWEN 2.5
# ============================================================================
print("\n" + "="*70)
print("TRAINING CONFIGURATION (OPTIMIZED FOR QWEN 2.5)")
print("="*70 + "\n")

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/qwen25_7b_lora_results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir='/content/drive/MyDrive/qwen25_7b_lora_logs',
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    bf16=True,
    dataloader_num_workers=0,
    seed=42,
    remove_unused_columns=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

print("✓ Training arguments configured")
print(f"  ✅ Model: Qwen2.5-7B")
print(f"  ✅ Learning rate: {training_args.learning_rate}")
print(f"  ✅ LoRA rank: {lora_r}")
print(f"  ✅ Max sequence length: 512 tokens")
print(f"  ✅ Dynamic padding enabled")
print(f"  ✅ Class weighting: {'enabled' if class_weights is not None else 'disabled'}")

# ============================================================================
# METRICS COMPUTATION
# ============================================================================

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', pos_label=1, zero_division=0
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# ============================================================================
# INITIALIZE TRAINER
# ============================================================================

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Use weighted trainer if class imbalance detected
if class_weights is not None:
    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
else:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

# ============================================================================
# TRAINING TIME ESTIMATION
# ============================================================================
print("\n" + "="*70)
print("TRAINING TIME ESTIMATION")
print("="*70 + "\n")

num_samples = len(train_dataset)
effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
steps_per_epoch = num_samples // effective_batch_size
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"Training Configuration:")
print(f"  Total samples: {num_samples}")
print(f"  Effective batch size: {effective_batch_size}")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total training steps: {total_steps}")
print(f"  Number of epochs: {training_args.num_train_epochs}")

print(f"\n⏱️ Estimated Training Time (Qwen2.5-7B):")
print(f"  T4 GPU:  {(total_steps * 2.5 / 60):.0f}-{(total_steps * 3.0 / 60):.0f} minutes")
print(f"  L4 GPU:  {(total_steps * 1.8 / 60):.0f}-{(total_steps * 2.2 / 60):.0f} minutes")
print(f"  A100 GPU: {(total_steps * 1.0 / 60):.0f}-{(total_steps * 1.3 / 60):.0f} minutes")
print(f"  Note: Qwen2.5 is slightly faster than Gemma-2 due to efficient architecture")

# ============================================================================
# TRAINING
# ============================================================================
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70 + "\n")

training_start_time = time_module.time()

try:
    torch.cuda.empty_cache()
    gc.collect()

    trainer.train()

    training_end_time = time_module.time()
    total_training_time = training_end_time - training_start_time

    print("\n" + "="*70)
    print("TRAINING COMPLETED")
    print("="*70)
    print(f"\n⏱️ Total time: {total_training_time/60:.1f} minutes ({total_training_time/3600:.2f} hours)")
    print(f"  Average per step: {total_training_time/total_steps:.2f} seconds")
    print(f"  Training speed: {num_samples/(total_training_time/3600):.0f} samples/hour")
    print("="*70 + "\n")

except Exception as e:
    print(f"\n❌ Training error: {str(e)}")
    import traceback
    traceback.print_exc()

    print("\n⚠ Attempting to save current state...")
    try:
        model.save_pretrained('/content/drive/MyDrive/qwen25_7b_checkpoint')
        tokenizer.save_pretrained('/content/drive/MyDrive/qwen25_7b_checkpoint')
        print("✓ Checkpoint saved")
    except:
        pass
    raise

# Save model
model_save_path = '/content/drive/MyDrive/qwen25_7b_lora_model'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✓ Model saved to: {model_save_path}")

# ============================================================================
# EVALUATION ON VALIDATION SET
# ============================================================================
print("\n" + "="*70)
print("EVALUATING ON VALIDATION SET")
print("="*70 + "\n")

val_predictions = trainer.predict(val_dataset)
val_pred_labels = np.argmax(val_predictions.predictions, axis=1)
val_true_labels = val_df['label'].values

# ============================================================================
# CLASSIFICATION REPORT (VALIDATION)
# ============================================================================

target_names = ['Non-Abusive', 'Abusive']
report = classification_report(
    val_true_labels,
    val_pred_labels,
    target_names=target_names,
    digits=4,
    zero_division=0
)

conf_matrix = confusion_matrix(val_true_labels, val_pred_labels)

print("\n" + "="*70)
print("VALIDATION SET - CLASSIFICATION REPORT")
print("="*70 + "\n")
print(report)
print(f"\nConfusion Matrix:")
print(f"{'':20} Predicted")
print(f"{'':20} Non-Abusive  Abusive")
print(f"Actual Non-Abusive   {conf_matrix[0][0]:>11}  {conf_matrix[0][1]:>7}")
print(f"       Abusive       {conf_matrix[1][0]:>11}  {conf_matrix[1][1]:>7}")

# Additional metrics
tn, fp, fn, tp = conf_matrix.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\nAdditional Metrics:")
print(f"  Sensitivity (Recall): {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")

# ============================================================================
# PREDICTIONS ON NEW DATASET
# ============================================================================
print("\n" + "="*70)
print("MAKING PREDICTIONS ON NEW DATASET")
print("="*70 + "\n")

print(f"Predicting labels for {len(predict_dataset)} samples...")

# Make predictions
test_predictions = trainer.predict(predict_dataset)
test_pred_labels = np.argmax(test_predictions.predictions, axis=1)

# Get probabilities
test_probs = torch.softmax(torch.tensor(test_predictions.predictions), dim=-1).numpy()

# Create results dataframe
predictions_df = df_predict_original.copy()
predictions_df['predicted_label'] = test_pred_labels
predictions_df['predicted_class'] = predictions_df['predicted_label'].map({0: 'Non-Abusive', 1: 'Abusive'})
predictions_df['confidence'] = test_probs.max(axis=1)
predictions_df['prob_non_abusive'] = test_probs[:, 0]
predictions_df['prob_abusive'] = test_probs[:, 1]

print("✓ Predictions completed!")
print(f"\nPrediction Distribution:")
print(f"  Non-Abusive (0): {(test_pred_labels == 0).sum()} ({(test_pred_labels == 0).sum()/len(test_pred_labels)*100:.2f}%)")
print(f"  Abusive (1): {(test_pred_labels == 1).sum()} ({(test_pred_labels == 1).sum()/len(test_pred_labels)*100:.2f}%)")

# ============================================================================
# SAVE RESULTS
# ============================================================================
print("\n" + "="*70)
print("SAVING RESULTS TO GOOGLE DRIVE")
print("="*70 + "\n")

# Save comprehensive report
report_path = '/content/drive/MyDrive/qwen25_7b_report.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("TAMIL ABUSIVE LANGUAGE DETECTION - QWEN2.5-7B\n")
    f.write("="*80 + "\n\n")
    f.write(f"Model: {model_name}\n")
    f.write(f"Architecture: Qwen2.5 (Alibaba Cloud)\n")
    f.write(f"Parameters: 7 Billion\n\n")
    f.write(f"Key Features:\n")
    f.write(f"  ✅ Extended LoRA targets (gate_proj, up_proj, down_proj)\n")
    f.write(f"  ✅ Max sequence length: 512 tokens\n")
    f.write(f"  ✅ Optimized for multilingual tasks\n\n")
    f.write(f"Dataset Configuration:\n")
    f.write(f"  - Training samples: {len(train_df)} (100% of dataset)\n")
    f.write(f"  - Validation samples: {len(val_df)} (20% sampled from training)\n")
    f.write(f"  - Prediction samples: {len(df_predict)} (separate dataset)\n")
    f.write(f"  - Class imbalance ratio: {imbalance_ratio:.2f}:1\n\n")
    f.write(f"LoRA Configuration:\n")
    f.write(f"  - Rank (r): {lora_config.r}\n")
    f.write(f"  - Alpha: {lora_config.lora_alpha}\n")
    f.write(f"  - Dropout: {lora_config.lora_dropout}\n")
    f.write(f"  - Target modules: {', '.join(lora_config.target_modules)}\n\n")
    f.write(f"Training Time:\n")
    f.write(f"  - Total: {total_training_time/60:.1f} minutes\n")
    f.write(f"  - Per step: {total_training_time/total_steps:.2f} seconds\n\n")
    f.write("="*80 + "\n")
    f.write("VALIDATION SET - CLASSIFICATION REPORT\n")
    f.write("="*80 + "\n\n")
    f.write(report)
    f.write(f"\n\nConfusion Matrix:\n")
    f.write(f"{'':20} Predicted\n")
    f.write(f"{'':20} Non-Abusive  Abusive\n")
    f.write(f"Actual Non-Abusive   {conf_matrix[0][0]:>11}  {conf_matrix[0][1]:>7}\n")
    f.write(f"       Abusive       {conf_matrix[1][0]:>11}  {conf_matrix[1][1]:>7}\n")
    f.write(f"\n\nPrediction Dataset Results:\n")
    f.write(f"  Total predictions: {len(predictions_df)}\n")
    f.write(f"  Non-Abusive: {(test_pred_labels == 0).sum()} ({(test_pred_labels == 0).sum()/len(test_pred_labels)*100:.2f}%)\n")
    f.write(f"  Abusive: {(test_pred_labels == 1).sum()} ({(test_pred_labels == 1).sum()/len(test_pred_labels)*100:.2f}%)\n")

print(f"✓ Report saved: {report_path}")

# Save validation results CSV
val_results_df = val_df.copy()
val_results_df['predicted_label'] = val_pred_labels
val_results_df['predicted_class'] = val_results_df['predicted_label'].map({0: 'Non-Abusive', 1: 'Abusive'})
val_results_df['actual_class'] = val_results_df['label'].map({0: 'Non-Abusive', 1: 'Abusive'})
val_results_df['correct'] = val_results_df['label'] == val_results_df['predicted_label']

val_probs = torch.softmax(torch.tensor(val_predictions.predictions), dim=-1).numpy()
val_results_df['confidence'] = val_probs.max(axis=1)
val_results_df['prob_non_abusive'] = val_probs[:, 0]
val_results_df['prob_abusive'] = val_probs[:, 1]

val_results_path = '/content/drive/MyDrive/qwen25_7b_validation_results.csv'
val_results_df.to_csv(val_results_path, index=False, encoding='utf-8-sig')
print(f"✓ Validation results saved: {val_results_path}")

# Save prediction results CSV
predictions_path = '/content/drive/MyDrive/qwen25_7b_predictions.csv'
predictions_df.to_csv(predictions_path, index=False, encoding='utf-8-sig')
print(f"✓ Prediction results saved: {predictions_path}")

# ============================================================================
# SAMPLE PREDICTIONS FROM VALIDATION SET
# ============================================================================
print("\n" + "="*70)
print("SAMPLE PREDICTIONS (VALIDATION SET)")
print("="*70 + "\n")

print("✅ Correct Predictions (first 5):")
correct_samples = val_results_df[val_results_df['correct']].head(5)
for idx, row in correct_samples.iterrows():
    print(f"\n{idx+1}. Text: {row['text'][:80]}...")
    print(f"   True: {row['actual_class']} | Predicted: {row['predicted_class']}")
    print(f"   Confidence: {row['confidence']:.4f}")

print("\n❌ Incorrect Predictions (first 5):")
incorrect_samples = val_results_df[~val_results_df['correct']].head(5)
if len(incorrect_samples) > 0:
    for idx, row in incorrect_samples.iterrows():
        print(f"\n{idx+1}. Text: {row['text'][:80]}...")
        print(f"   True: {row['actual_class']} | Predicted: {row['predicted_class']}")
        print(f"   Confidence: {row['confidence']:.4f}")
else:
    print("  Perfect predictions! 🎯")

# ============================================================================
# SAMPLE PREDICTIONS FROM PREDICTION DATASET
# ============================================================================
print("\n" + "="*70)
print("SAMPLE PREDICTIONS (PREDICTION DATASET)")
print("="*70 + "\n")

print("First 10 predictions:")
for idx in range(min(10, len(predictions_df))):
    row = predictions_df.iloc[idx]
    text_content = row[pred_text_col] if pred_text_col in row else row['text']
    print(f"\n{idx+1}. Text: {str(text_content)[:80]}...")
    print(f"   Predicted: {row['predicted_class']}")
    print(f"   Confidence: {row['confidence']:.4f}")
    print(f"   Probabilities: Non-Abusive={row['prob_non_abusive']:.4f}, Abusive={row['prob_abusive']:.4f}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✅ TRAINING AND PREDICTION COMPLETE - QWEN2.5-7B")
print("="*80)
print(f"\n📊 Validation Set Performance:")
acc = accuracy_score(val_true_labels, val_pred_labels)
prec, rec, f1, _ = precision_recall_fscore_support(val_true_labels, val_pred_labels, average='binary', pos_label=1)
print(f"  Accuracy: {acc:.4f}")
print(f"  Precision (Abusive): {prec:.4f}")
print(f"  Recall (Abusive): {rec:.4f}")
print(f"  F1-Score (Abusive): {f1:.4f}")
print(f"  Sensitivity: {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")

print(f"\n📊 Prediction Dataset:")
print(f"  Total predictions: {len(predictions_df)}")
print(f"  Non-Abusive: {(test_pred_labels == 0).sum()} ({(test_pred_labels == 0).sum()/len(test_pred_labels)*100:.2f}%)")
print(f"  Abusive: {(test_pred_labels == 1).sum()} ({(test_pred_labels == 1).sum()/len(test_pred_labels)*100:.2f}%)")

print(f"\n🎯 Model Advantages:")
print("  ✅ Qwen2.5: Excellent multilingual support (including Tamil)")
print("  ✅ Efficient architecture: Faster training than Gemma-2")
print("  ✅ Extended context: 512 tokens (vs 256 for Gemma)")
print("  ✅ More LoRA targets: Better fine-tuning coverage")

print(f"\n💾 Saved Files:")
print(f"  Model: {model_save_path}")
print(f"  Report: {report_path}")
print(f"  Validation Results: {val_results_path}")
print(f"  Prediction Results: {predictions_path}")
print("\n" + "="*80)
print("\n✅ SUCCESS! Your predictions are ready in:")
print(f"   {predictions_path}")
print("="*80)